# 🎬 Montage automatique Locapay

Chargez une **vidéo brute face caméra** : le notebook la **transcrit**, **coupe les blancs et les hésitations**, ajoute les **sous-titres animés mot par mot** (italique fin + mot clé en gras capitales), les **recadrages**, les **écrans de couleur Locapay**, le **logo**, des **effets sonores légers** et, si vous voulez, une **musique de fond** qui baisse quand on parle. Vous récupérez un MP4 vertical 1080×1920 prêt à publier.

**Mode d'emploi**
1. Vérifiez que le GPU est activé : menu **Exécution › Modifier le type d'exécution › T4 GPU** (gratuit).
2. Menu **Exécution › Tout exécuter**.
3. À l'étape 3, cliquez sur **Choisir des fichiers** et sélectionnez votre vidéo (et votre musique si vous l'avez cochée).
4. Le téléchargement de la vidéo montée démarre tout seul à la fin.

💡 Si un mot est mal transcrit (ex. « Locapé »), corrigez-le à l'étape 5 puis relancez seulement les étapes 5 et 6 (bouton ▶ à gauche de chaque étape).

In [ ]:
#@title 1️⃣ Installation (1 à 2 minutes)
import os, subprocess, base64, gzip, sys
os.makedirs("/content/montage/polices", exist_ok=True)
subprocess.run("pip -q install -U openai-whisper", shell=True, check=True)
P = "/content/montage/polices"
for nom, url in [("serif.ttf", "https://raw.githubusercontent.com/google/fonts/main/ofl/playfairdisplay/PlayfairDisplay-Italic%5Bwght%5D.ttf"),
                 ("gras.ttf", "https://raw.githubusercontent.com/google/fonts/main/ofl/montserrat/Montserrat%5Bwght%5D.ttf")]:
    subprocess.run(["wget", "-q", "-O", f"{P}/{nom}", url])
open("/content/montage/montage_auto.py", "wb").write(gzip.decompress(base64.b64decode("H4sIAAAAAAAC/8V93ZIbR7Le/TxFn15vTLcIgACGHJHgQjal1VK0qRWDo7NyGIIRDaCBaU53A+puzA8nJmLfwNfHVxsOh89qT8TxjcPhE+urnTfZF/Ar+Psyq/oHwJDDlRxmSAN0V1VWVlZWVlb+FFzXPfh6lRbBMnSCTbFKgiL6YRM6f/knJy+u4tB5tZoF6+DK+cufD9r3+Xfwu2h+++PKmWabInQWwSx0ZkFy+2MWOE77M8cpsiDNZ1m0LqJV6njfnUb5Osx8LUTL2WqzDp15mDvTOEhn+QHf56tN3i6iIsPrII0ALneSVeGsg0w+vagIYsF7EaXOA3k3i29/dMLUWWZBDhTWrBLmvsDLwlkwzzBmgIuLMEsJ8IGDFsswc96tVokzA1GidKN4AS2iDbTwfhOHm4xfkyBDlwIwXi1XWjVcLMIiB8bpitgqyFzLkk0uSKLtYpXOHW8lVEjDOA5bznwzO4vSpWL49etHznmYFdEsiJ1e90n3sve033XW2e2fCuf2D856M40j4Oo92KJop7gsnLBwOnlW+J0DF/MbJetVVjhvgVMLSBenLWeVt0CElpOfboooxudmus5WszDH+4vgPDxYZKDBPCiCWRzkOQZigJSvWiB1GM8PLPR0k6yvHJA6XWvj1y9f2UYvE5C6pR+/zoIL8/U3EWlvH0Dvg4PvWs5XLec3r0+coYy65XDYLeeoe3DyBu8ePel2uwcHB79whj/hn5NhVmJO/8G/KQd0IH+dN6GWDA4c/DOzPekNsBwyIOD+ov9l/4vHPdep/fsFeDXcgARRUa4Xz/LMA/Kk34DWr6B1n3/Rf/5oCxon3gKShutVHnF2J1wIE10IA2cRr4ICULqd42Nt2MXTKZYxyIZv0yCX1ul0EgouE4PAwIlSNuxLsXLsxHBsiZpyrltDq3y12jhusJltUlcAnK/iTRJODHfX8eo9kQr5Zr3OoiTMJqchBhJwKOhoulrFqPZthiUhHYQbsObpJmkB9dNOp6NjDzY5gAeXdcCPHjvNf7+A9GrPw/j2Dy0HUktkhxPmBWm+hiDwcp0CrNllOAnOg7RoINqtl663yNt7LKXzTRaGkzVAN9Hpdx7dgckmrUSNYBOlOUTh7Y8Cj3KmgtKz3YQQVBAKaNIgkZRRzkzCFBMZpbuliuCCRRVUnYJFNDuFvJgQQDnFrmGuOJqFkzzMosX+IorQPKyVHRzMw4VzGl72s+XUO/V1sZyS+zqoFK09vxPrF/cXrlI+C4tNljrFZh2HHvjPOx1FgwjLoz8Gux77EImZE4FCjocF3285j3z/p6/0hnAUrCfYHPLJPFxvIjB9uEzCtMg9+8UMhXUwmtFYiQfMUIG42XpazRZemKLOMiw892KVzXO3hda+g8LRuKrMf5wW0Ma7GElNd+xbmjWqRQunaDa0iHWC9TpM5961e+EOHCx2N3fNjBNoXgRZAah4H9bfowne3jRmg+DMbOIrZVODYJ4hB3aRJL50vK+DmfN8jQl0TiJwxipFJ4sghxRvX+hGrnVepkUYOw+d11+gwgrIBlFV4YtVHEz9jmt4zOwR+hFH047uSFhmBUibHBha2OdOfoX+Es8HXzjur4PsIkpdKAXzqkYSgNfT0FQJsuT4kdZo9tHBQplPgNPM4+gmBkHXr6huqMRi1yJyNxClxPvgaI17gHoPDCWnwsiCKA+dNxuoKkn4ZZatMs99TsFsZpOaRlM9gACCeKFq5GFQdlJ2ZhESvjltmC+uxq+/+fWXr748mXz96t+Di69doDRdgclIofZslSSbNCquHppG7ZjytH1+1NZ65NQEvd/dQIrbJHdrm/XdJJxHm+TutlqujbFXma7vgRxb3JRD+83zk2+/fNMc3XvGoV9aDfT02xYS5fcbs+DsxGShdx7NwxV0sxX2jXDo2l6w0yw3eF5keDiHPjDdxAHqD123WpjfNubX6NMOdLiZc7rKVtRvsPk0tGXDHLPTVZRHkFvYbtCXY9ZNtTRNveEdssGysa021IVSsaxZ2rXFVRZBH8AWsNjCnjzZcq6VDjf+X3//3zDwRbzJT4fc5CoBKUhVYDsWs2lJyzX03MkKGsdikoXr1bDGuSKltQ9Lc98QG6QaKtV3+G/fP8rvCdceFlWyzgXJFtYYlLUgnkCjTtbFsDZx3A5+u0rvBxzida5aH/6DSnIeifIXXhbD3wRxXqeGiob9O1uGHcF8h/jfM2dGKNV2NKrvTWFmp9Kw19ck2ofmsilS7jWtCaa13oXXXJT7J24OysywJnh8BVws9DWOnpPiao2XwOyJW3UASuDkMiH37OGZbRb4fze/P9PcNroZXRvNY2C/4ttFh18osEQr4Av5RuUg1Qr4vKm0GC+XFrlqLv5Yi3Kj4OSWg5QdilU2O62/qC9zzAu1xdlmHriizbByh48djAEqeAQ9IOY2HWLEqLjelDujtB3qO9m6dbYFwVI8qkhtyNlKDtd2TuVP96//+Q//53/9J8d5HchB/sXrv3cGhHv7RwA+FDl+6HAjjnJuj2vwJ44SqZymE+jbOU/fP86i3O847tYEu89nRXQevqPGr4C/vETdjayEzxwwc7TgcR09kS+d+WFYL//2EVt1aoxqxju0G8xBfaF9ccrxkgWg8pdDsKIfphu71GC1weErc65BzxsuO3MIAlwrOaEdzidSvVxZZkHhw+iKP2m57F8K91hDzmLdOx56JSuQj/w7wH1oPbV4pp6u8rC+vO5U8hsi856a/q6OX+r3/9/V+6++PHn57fNvX37z2xPRa3DO5ko5DTf8wJM+qs5ymshHkshL/YvjuHyYhuEplJiTb795zZMM6eC64D9oEDEtXlwDm1RteFhn4NA5lxBUygTqqjNf4WQOSqYRrHKZ8xYrYuNEsUMjmIPVkGLmoOzgTxTn8jY/SFApRFdch5AjkRNv8D83MP7JnSsa+mhpRD/4v+C33Ln9xwBdODPnB9Sm3e2HDf5Da6AAfG7/hTgFtKYF+NxcHvCEnrMsh5TFeR0PSUJAfypoKYxo3grOYbugevXOYU2Mc423IiyK7PaP/Bvyz2qN0zMQpNIlylhOYxAXI204B3M+zU4B5ZzmwQQDL/A/egeVnCJAdYdds2Oa39IV4IIs/JuuSCC8j7jZgTawFwAG7CR4M41CgtoUBwVpyG+h/s1xYAfzrkCMFN2cryLYKOQDE4dv+IhmEQeLPjDMKIMSAIvWQlbl7FCoc4iZ+mFj/nJyDt4egjCplM4PMfX8A2Y4RBcYexJhahQgTC4OTCukkVhYHN3NHDI7y8J0BgMUbcBvgc08fBs45zjgcBkevN1Al3CS2z9htAkkH3rnriDDR1NSWmZ2HW4K/rkEWfUPsIr4fxaCSzv5Oo4KD4aF3zx/+TkUi7+FgckvhmPIdnM7r3umu2K8A8BLyexvP8zuu5w8+ygGObAMcsUlA+YCzbh+ijqHCseWHFSfqZIrGwTTMwtMhIlXGGFnhEwWdmA+9jJ39B+/v7j9w+1/uf2vtz/e/vH2T7f/dPvPt//99n/c/s/bf7n98+3/vv3Hw7/+/h/aY4oQ/F9g77kIMwDvQEOP4SnAHv37f2DpoWs7hCwUw34Gk1cGW5qYDnmAyanplW+q41DtJXbhw5hW1Nsfh8aa+ozsBSsfxrjJhrXv+aE99BRUSigmb8ptYS0rINKhCkEw2Gffp2O3gQSlmltXPCDj3aHLhgKhKe4DWDnRjZQYqKgMU9jOPhHYDWR3vxBkRzIrZS1/TNtvY9OJOHOF1t6xKMgmwWcIieZ2mBB1FldtzlBDukuwu9nzhOnijNW3+lArYjojZIyTJg6oM6Rfq/N3//rZAErJ+MG/wrC34Vk9ZR7NoPNDtxjqWM/GMBl6AnLUHYtlSMCrDun6FQQMx26lSWNXpHA0R3AoCOHWsZbjbTK467idt1AAzZibpPkZbJS0KIs/Sf1wghkUGkzgBrwhBmfD8WLgbTlvBqWbYsteqdTymyjKeQ8KAlD/jF+g1Dhtpxe2j8aWO9509tvoD+qKCTtIdkHX2cFwndt2feE4VKspH+M6Mzb5Si1ZvwviTcOOJZJpFcuuUGCZ3f4YqsCFrD4XR6NVmu/gVzNuKLDBpacPLUuDBzDtd59gAtVsD+clTPKpCn3TrQdxqx7JuPRIgoilT8I3hny2aiydqKXYhPCLhVlQhHW+MlTQZnK6sZMir0bt3njkYqnw0IfXvxpifio3SGOJ7G0wNMN+Vi9m/9BFzYqIqnXClbMHaKWGzsOpAB4YPCGgbV8Dx9JU4ePoGY1vqhl5RxoIuKqHdyM3KKcEM9DiG9sHifCmU/PS+PV2U22HpWhWwrv6sB9ULalfGCw2WzOzDyVMButxKgx2oDneKOWmllf6TTI1Kuhw6q9aBmP/WVVTJ8F5MGSZPrxnHtDMzsK7JqehpBoQeO1dk9ekmt88uYhrpey12RHfYeJGLhvKYN5F9WVBEZQ73j63ecusD8oPmJWpjsluCdVC5JpowjAwLo0L9Gomu+uox4l/06EPDHtep/vIPimpjWjSjj84f4EzNBP3zInml84+8rKt7FCQ9svQ69FPmHqo7W/tq4kcIFuy+yhh5pejM0rMMWVH+WY83t6oy2UcQEa86TT9hc4nGNenjy2TTaWiafEZi3Z395lsyJ4ipILsgWniw7vS32kg5LIsM4KGAQjkP9IHXw/urKcIme23JPo1KDqgooJCeBvxyQnCV5nFUeT8UmgoT9ZSBGb0qNr4TY4UmL5STJ2q3Uq9Eg2JFWo2m5G7EkYsnqE+1svakmwt82z86dmaiBZ1Xl1m5EcR4rWwEalgikiWTJiq9bEiGw23dJtMrFzCFSjkKm+yRTqDsYbHjE3ikVhS8+1Yt0vp9y07RVufEShh6u3ThNC1V3JXxQ3km6Mu9U7qkCF12aZqRc1Kq++ALNWkTCeRCHyGCAs5mwvSnzl9eUpKufB3QwNNn++CZzbddkWW7nhsOR0e6T2KrJkay5PE5lk5TQfVgsi2NrDtOdkDx/CGOalju8DBdpOamnJyw9M6BJ+odwT2M272f/lzi6Y0nMPMNwD4y59BVB/nChzg1vSJ8xxYAoq4X5XstIwqYUMCG8x8CpIaAZYtJ6XAMsWjZQSurB7AE71q/DDaUSkRcD5pKdJEFTCSein8Z3gLvZtT7ja1tVHxPqbZAbcPRqOS3TrMYkgvddL3c0rZ2lbLK36xAAntV8LgW/ySXnZgjUWMDAMFlp31ai0HVBa9aO4VS6xp7geNVW0nogJ6oaqtboLVZomWFeLvuKmINnWRA2U7sKoCNBga/bSwvvGkl4WVEFvzWg0dbKzvMOIGs8ixphc+rbYxIKYqEOC22J8oFqHRUY5LraNaNCC2DFD4RpjlQucUNsQzlUFnLbVsVlS6yPeIC6vtm/ZWxRdTIKd0pwORKLX1K3s/FSVByh/1ZOiKoQyVlt+y+otKBzXa5QWEN3VFfMVfVURpBI0Q2eYCOvcmmrBdCXjCE+HdNFT8PbF8XgKjPwK45mF1rigXIWtlhaFGatVWeoEK701nJ7bKr07cJUleCB2OK4Yr0L5giNYL4RYOCKv+RanH17iHtOHkRRVfqkzptyzsttP32aGERf2KQCMLBmX6JF2wsN/p1oBH01j0yXTdibGm1jTCcMW1yEroIfV1JBQ1Mj8jdjKuu1bypogmljPZ0QT2lll8xt26C06dQR3yQII2HpuMhnCITE9TtSELFWgKge0gmOZexG1QJWClQQAXf0cnI4sqyN1NB2+Ds8bbKc2Yury0EVZIeAUHRzKdwzwKZQide3WSPqiTW7Qyjmnm+1tqQlTuSeyjtjqVbWTANqDqTaeMO8QS3X3b97fU2pZSqVq9OZxx4dwTkmxJT0VX1wcVLIsAVNxfIhbroG75KNedmh9E5vFRPvEsShue5RPPZsngzQuuR6pnXJH8bDpu3GKFMBRbJie3MnjNxkfk2awPU42ojlguZhgNhXFkMBg3lDTVENXEcUyeLyBVVX9sksKM0uiaiKRtqpemWETSTw9Eo6lJA9A0qs6D3TGh3MJ5SeJ/z4MsErMLQ3NhDzQDLrKr2kEOE1aGy8IltwnpTtwCVTFXo620x4ZfTMqeJtOrSRok8OjZNzWzwOUsXBfwY/IDJVt6LyNmt+i4OHh/O1NtD/4y3gp9nf+I9Xi094R28AcaiiA6DDtfpvbTBT1x0+nqUmuZE0xAmXmkRwyEGzMyMcPkxgw3hORBOVu3ebSwL9QEmmDRWxp30vDCc19BG/IIA0vRiKoykLnDP5408jvs3/PYdZso6hd28Agt7RAw7OGCsdNxPOw9NvBsr/IJGynP1l4tRLrzQnw7Qfp5vMm8p3aLSZqIvnnx+fMartyjzH8+dOko6SAsIojXp4FBWLlLlJN7Qtk3eAC4a+h3DNvrP36MAM/6n3JEHUFwwhgOhjyHAr6+GKOkpbOXaVcyiWAYDd5+LSvLSHpZaxM6micTxHfG6P9NTRayGIeZBG5575PZqb9XNKQysFltVzulhFnlHcYXdcLLKEcYxoya1OwUe3k9bgMsPaFYt524D08kdPHhK+wsQXb1kGsgf3hCw4k484P44YtwlS2jwHnJfIZZpygW7vtiSdyHmzx7mOOwFj5cCDS7pB7Sf3a+efhrfPxuc8LY3nYN6Dae04/E8znERex8vornPx+OEOTtEmI9ZCdedLB/mC9T8yWzbzLiDvtoLYq5VT1r6HLLTof9Mm2CnyHyzfh8Stahg1+5RhJhSunU9IV4zcKmO8Sox1UXg23tRHNsGN4wdXf1k6oh3R7DcucoKWG3DvfLSzgxSDy3pEpTAQnjRm+zj+4tr/X2tYTYGCYte8y3e9w2Z35cJ3dDN2uzAcoGMcOLwPj4NWSUBx3kTcvEsNc8hK8RxdNGWs1soz5fm1zBU0QQS3YOzB9024o5QBId4G2FLU6dzhpdS89kkAW3/1xU8ZJy5OSMv6jpI1hJWyoyD3uVBgYFmhpGk1Q855C9ltaM2eJXHnB2FNyz/RBUlV4ax8TInMgkmKTF0JTcmp/k3HeRb5mt9s+eOvF5lPwQuNFg2z5anf9GXuUybnWeDdQBSJcw2p0Ze0MH4obOYTE+liXsZheshhPcC60zHrIHe1CraBVLg1hizWTALT1TqlOBJI+bzXke25piyw0ClZAU/C5FWUlZflIyP6Qx+/aY/vSEyoN3/KTlfNrnt8f4dizfHuHb476/x5amW6kSpKZElUtBYpfyot4dZYMRQ3rOszhxZyty/33xjDIESknWLMe63zgY5mugxaNzt9N/AsWr7Gdv9fjCWE4jnpP7ddtMktNeSnifqEFM3rS33ee1hQJgOAt8J3b4J8eDO4e0eyosJ9ke4dgZjt8XLTEGCHJHW8i1iNvWKRDuk0wShh41T8WnZpSZHWJmj/k6SGlmhmne7gz0ytD1K1Tk7rebMMZG6OrhtuNA0LbjiU8F8bL/XTLRWOl9R+3rwt8FZgF6paJ2QU3y1LfMgZ7u4FgKyhqBW+iprSCu6PKPT/EIQPYluHL/VF/SXXBxodyxU+OKpYBlyPoe75uu3bpU2REopXjeXfnK2LKQn8Io9wTGlE+xno+7exbsBxery33QqBi6IO2qe++Ka6y243K17ZLt51ln91xju+srPjVmtb0rqVH1siWc3mBA5IzKJ1/dwd8fx44/CyvuZ0NwjEAX04t8M5rLJAzy0Ls03HFpjFAkyiWNcV1fvKSNsxCtZx7/XPrOJ584RwbSbIU5zibMkw5pZz0PsLZlEEAfVDBd0FDCyepuH9D1FE/dTFBa4/CYz8Rl96QvNuYeBXeoXkUiKa5c8kevc1xaP9GExvunT59W8FPQPrVz3VMTKs58F9EccukTNCFzN8sQsrg8LUxhLbGkT/IlCBzLo3eh5ylk3+QSdz5/+erlb798/qZqIbNRddYGLso8z1Qi1PpqE5Iy1ANBo/eIvE9ChwaHprCw2NihBxh5b7e4D/vd+sqTs3i/OozzAWscKicTwRH34voQ3+zXWB3PJV3XOwcSgW8Q0GndOS0DFjQD1r70lYRX/s+RRbmV1O7lV2lxiogcRspjgU6ydKlGZOwZ81XSARsGm7jge+9Ta8+ZzKclf1sO7pJxvUuaTbtlPfGsTaYAFXq4RiCCmFuAwxbgih+GvU7fwKAtngJJqjDIYqY4LMNVopZs2ypFqQa3ofhdmK1yzxi6EIEoidJiY8cqQ3ygJM12gdEPB43QCbW6p37TGEfjEVPqO7msVnxZR3i1mNGT9NA5qbHgKbiLnRFdlral9zZ6/EQwKCsKOuDJBQrY6JlUNC9YKKMhiGHVcCe2LV8wPWq1yk8ZLDPsdmBqWUczfKE2uegO+4+FPMPHSKkHjZC+EGyG7d5RjbzkIDRGtxjIM6fY9hRAJpG6uibScy2+OA2R0iY212gGdixAB3wTGdXvkEFlLRU0lss3FkppKT9yORnW2ICcxMQVRA8ixYAupiD2JPtUxuBTIKXndULkhA5saLBHxgalCjcymISfsjaZUYfsC1nwCgOtkY5exZImj7ZoAhnY2yJKYPlDZv2g5I9jejuedsHorBVerr02CQJe6z2pBosiMlBfawkL4XO2SbgpLxSmvwvi6LFvRdNWSbdnjWo1emyTY4sMjMmoCADtCMpClQNsx45J+vDQc6NP7BkXmbjYHosnviszFCbc4/sZRBeL5Mjm72zpcHosmk6PUe+o94Rs3nv66WN+9o+PPsXdEU8f98a+pfUnQ2cfuY7uSax272mDUsswBbdnE1DMC8/xIFkjUAno36BRkIb5LQp6UkjLs28p+aopm545b7I90gqd5ZLAoesa7pNqkdMbs1qbV/hG/4uZQfPSPt7UI17grk3oXGlSshpJTdzlEWIVVgtyNRFB1HEyriW15OXoQF5UM2OrWz1yqgRNdQtAySoRnP75ANFbhNKtlHLr6sbx0kTHoGpNEVgHJKpXF0ptkXTyJwQmuVkhTLemg0uPt+2er55NxJvx2GsIqoUl2bV92dmrUT4IJXiP+A6kdKyMnP/AaIROycLAy5dZvG+TB9pEmdSsq1mMUFrWxA0lZ97oFcxZGe9K8EcDElq5SIiMt+1eGTp+EUHD4S0uHeZwl84h92KKqFyEwFzUIh/oD0qN7pF7feAsr3Lkb4mmVL1aZIHhjvqsIr8wg+YhhfCM5lTVjvqfHn/qd5D+Sd8OBgBsccFDp1hNr5Cu4P0cGkkG1Xyj6oIxIQQL3nERmuw0w7e661e323SyTerBWbvA81RSB9vnktHEmGN5hFURccYTxDGHQUIrkns+6GrJ6epiwhSJCB7O91q8tfFQCNhSrXKg7yY5kJswJ3uYrTTI2pYg9DvXl+H7obdXC+LD63yYFs7RYv6R3gDRFU4wYqiWJlNVEuAkxRZMNF+ZCECqOGwtuX+5h9f+yLUDZlBK3W02x9mX4+D5F991NNZBuirK5Srpc3OKkbkeWcuRTpBQWey/DoMJEpYOkieRz7fcMNKDZD1KnlsOFMoGVqhrJf6lTKPKohiQouj3+sbXZ0tcuLJYs1ueVSjnUehDPvSedCkQntbElCEEJOxFfY/ga7MJqOtaZqLBfdYvqqh/DBM2WA1RLlR4iiH60ZHfI3/cssksPx+uh92P5pQym8Qm4kzmG94R9LMvr2DviM0SQrpIeHmv5fBTx1mfMN6rI8viZ8jtgLPB5HVoSnEkwR4aY4S9xXohtQqi+LIZlDGc1BnBEdFxRTNEPKSBv3JhPIdmnDBNaQciM9KYNpGHG9qs5b6wUFPT4jLDpH7TW+fAJJBox4ht5HIVL0i5cyDSZVNM5MovWBR9k4FxiORWSTnbxBvnmZYJnhQvmxz5gZpUiAK9yekL61WRruQ+JI0o14vadq87471fTvSQnj6mvhVyddIKl4A5Z1+9wykUwZApcuWCt5grc6eSJCimh7iJTKI9JQiY5wTJNGSsUzlQSfSEvwizvZEr1fjZsRRuZqpok3tkq+gsSIqgmaLMJqjMV3DhhxIKLFOLLd/tWNDaH9zKSXAG51qWe6Y6lBj6mCers9pVCXA8AcBuxkl9wmSy/O14Zu2tptSpEjFsiDCjtG5FNpbZGirO+FKEqL8VmqgQmVFB8xrMn5rMjdQs1a6ozykIImg05YaayIZt6ZUWpK2kix3SL9yvDUNdSwTljeNdW/88bkALJb7FDOmGsbvKkpgPBPyaZFfNStU0I7B6PRF/rSZ7C1HywMq5WbjX0aB7NL/pJOtHtUazRGL2IAiTdbgUAYeLt2Ko1XFTHF6p8KPIAywMedA5WtzIy0LflcSoSiLXHiwqky8XyLAS0ttTaDg53w7jIaIPiGl7ofconC+ie/S+cIN0E8eI0Bpmw+uTNzeDWTzkbR/hqh66WAI/X0ijHKs7HF5/dzO4/upmsOC04YqWaEm/6yTgzUcwGXGHg9yfZbQ+tmZIH7EtWgvcKXCNywFvWqKiZsNey+yMV5vzR/3ues9O4bYD7RzxB9RqiQAQbgWmpdF9J3FwhfWf21HsA5QEa1KgO7AaYfkiwAs54GAWNBGxJ+/2QZkNZEtEeOFlHxdTEQ4di2EhumaYXfHeFHk9y2RSesd3wJGdMwhmUnmqjxCbZ/IckMF4zRi0dbPBIiRQIJOla5zT3L0xacJds7Otu1mM0LFmcT42+Es3qu3FyncwocsRnod5TYEwYkjeGkOcbDp3rTNXinm1pVs74sjpRkp4uMHAmEE+xxWaQ3dTLNpP9LSzaKZBaeygjmU7Nk6OMrjHi0H2hy7PZFVm8qHkJX///SFykym6D79PDS571J97rXpVW5C8Guh058FCVKZuucx0bAf71Z6ZNl8rsGR1vohV33UfkIP0mgirR4x351Sv/uxkCSRgWJEaniTYeCeCbF6rvmeGzexuuyW0R2vMxTWkuIcDt4GUaeOJtRQgWBSMxBNjDyZInpehY0vpPDrHZSQenZJHx13+Q2GyW3hsivLdIgHZ0MLd69NBtz+HKEnMZy6frWv4gCjE7b2GoVzKRcxN7GnDilOxXnWwvgfvMZqj1+DEZRXFauNmx3tTxJuBsEsTeYxVvVNgY76Z27ubRm4DRbhvT5sP8AMFe/LP9A7ccO8yAT3Pbr5Pr2szHPg3CLH/zKm/m/qsdOgc6pK+GB1eHI7rHtRDelAPx6zGJfUMlHpAUtlbCVMGQ9pLvWSgtfTnSk82t48OVdvAglxCqOY4LBnFf1ePNg5WENiEDXrGrtEIItqaoEZIkbpiGMe/L3L7oNSvWmWmmkLT4OdxyzxqnHQz4bER3XxQ3f2peUpDfVhm6rgokyYkk7t+1aekIGxFKzZr1HTBeFmGggp/b1XE0FLq6CZGFDMlDeJlh5u0Fy/LSNyaTthEelk67I7oc6i5+o7EQo4Kxgn3kN/F8OD7pVvv1fPffvEfvjmpQaf35lPci0zXs6mvTQ0Yppsd65YM/aCJlaVeDSsb0WGQAviPw0ngzoWnCEamT5M6AjvdtZThtnPUqYwQ3Bk1S5JYD5q4GphMSzCPDWgB831Mjsue/BbeCMPFyIRIbL3qrtFaDDuwEkVM70zEtZKkVYtT8++UWbvxbFQ1yZNjixPP/JssNAk2ikXL3vSShXMuVaT2t2Tp8fQUH2IYp3JHCjLvZqtz3mv1Zybjweso1+Oium5jF7zFmv6i/XbA0h3BCfkOc8gGX/HzVMtOcOf0yVdmSxIP3iyMYi/n3NMPIEb6vnGn1spPm+V6EzKN4wu3pqeqxntCBfaEOi/36OE0mm3wv1ut2dqdv+XEAxgEodsKfxhSECOQtYD3s3s8yIPCGIL4/GSwDJIkgF/v6ZMWfP+rbBrwnoJwiP2bPqLBNB+25UuWDCWLfZrIi74iwMnRax5rmsxrkQD30GUia3ERYw91mvNFqd0gQFzK7mGwghYcXU4WiehCEKV9VY3JhGqoGdaxe/n6SyNfNcxmiGmkKZzRT0f2FuVCfCXI4oMDhOZxN0qWNiWsivPVoBvlSUkHH2ylWxLSSMAwHxGfzY1yulmII1qIaI1KOLxAs9kTbWMWOhr5AKY17pOYVEfiQV2TMEDr5TA77WBZgwEijEtBz7segcyCcd8Q64z81wUBhsd7nUPOhBZ2G/fTGM2qBlfJWiTryXnT0CFsIKdkK32QLSv2gr+F6WoKdMVie1noPoZSc/r+7uYSh0yBY89OWMO+ZXLwYuv9x7d79LX3fPdkG/fyNKuU1CUQpftXQDptqtPGK0TUsUW/bRjo01ogw7TG6GJATyHM0Kp+plvRiy/qyVZ4siyMt7WEVbNzmXuYsS+91QxXyayCdrkqjPN9ly3f7jL02mhAzJRvuguJqMnvQiDWymRsrXby5ZOlxNzUl3ZJInFSCX22wzMzcSfWkz6ogrc09cPsmBrgwyu8MfQp2ENUyp31KLBqe6H4JoDVncHaGtKzPxmHP4vAmF+FylyUHg+hkknj3yfsHv65mWYrlCluPI6LroMkudb29Qq74WzvlPByAcRYfJadrrkVhS5MduBLCFj3aHdoFzxD8W5X9PlOt993uwFz3VrQ28l3JpVyduFIiB8D4E6+kxdg66tG3a8k3vAxC09rtb+SF/7dlMZ8lKqfIbGNH/vi7/E/5n51OfQuIfWu8P+lJK5e8IFfoAVuq8Q7QQofPuNVCQE1HazbKznM6mL7Yyk1dteG9mlIYxVieHeM8wdiBJVPGGbK6AiEv+zsYZVaX+dxLv83nfqPFMgrQPtsWCmyfFWujzsWwz78qk41lpbz22eusqJr4SvSCI15T2zvPRbbrHfHCrPD1/NDbfgfMwxp3DJxrOULc7owUa3eV81CPYaY0vdvNioZRXIrOerUMHtvRzYVc5o3AYV1fnYbHvt6fm11uNarJOA35VaJVfhY7tzvbt+RotfWOiqQnev05uF1Or0Bmtc9jYzimPjml/4dN0OX5qcdfmPviBfa7daYp7itpSYbamvosxg3wjIgs3x/EfC2w2elOncG/cyrXx/Q+Ac/2jxaKeTz+l2yqtM3fleFd6eYH02p0JyuMk06Mfna10Vt05nUNh2J9taMFN6tc+M3LElSorCaJFjtWI/MNrxrHVq9n5M5vjIi2obLIByoxhkzvUpgtPzQSVHSmHbmcTclarwN2lCJT428/CX0e7k1w4bjj2wvJpkIETPmWC9X3cQiJB4iMXc8GhyPd61ys/jDlNzbWb7n4p6PoixjuJpkFc6iG6VGdl3YkBBVjFerTKWHwDY5Uoj6amrheAEWP3dtNFktdK0WtIYXhuXtdWM6j5r9Py5dkJRlwRTCdfcupbMPXkk210u6dMt7VyXdl8AtTRbuCA6WcQA3fTK8fjc6DA7H4o0a8GFqHlpwFhVr+G9ef3vSPvn2+Ztv8aXl7peScAYtgnk4LOBnGszlTNyybzDBiMgZXptbPua6ISMoVrqUykejAKbPcW27VzLUMK5X4PHQfWaukJShiQPhGR0MrnmtAGx9xqMvjGtgmA6vybRKv5vB+RDUgOPrDnyVCYhz/3EN5/7jBi1AACQuDMFPMJgwOvhapl4pGa82c0bYDl8y8nXw7ethuz949eb5sIdem060LZjvdalhhnjJ6bA/Oh+Pzs/Gz5qNRz1O8vsh6A9NMZT58Si35JVLa1XybFkj+ABeLkWy/YGqammZpuqkNHFY8UqWoNvulfYN07AmKMwUPRv1S9bsDupE3MuOH+WC3GHehWvGf/2m0/zJrZvWFj8f38UeejqEg6BTcQe+j5Jucz7MnOA1p4qhXEASTllkGnDnHxan+AD36OIZqNP2eBAUDFXExQaDDHE+oMHw6HF3lOyFDS7IxygLkugSeK858qOBhnZDIwc9bbjTcIGAiKKF1wlUlWwon+j4ydNRgPGVjNDQ8tR49ozoq686Ss+eaaf1Hvt/W4+lmSNoClhRCDrJo8D9Ca7BTwxfio1DLoCQFI+YkVDOouZ4Nsjc5SE0PvyKK/c6ht/rR5Yh7nMe/o0+z8jaNOoPQWvHnVl5q+8Y28f7OysVyiBguuZuN7j7xhQ4TxiedY6Mhp27Ub45kRCUO25G2ecOxarhupUrffnH9Lws99hSVzpoJiUv3F9vMt5+a5LUB861NDiU2eUm2FvcwB7tXVtrzOhQLCj0rJX38+yswmsG6esdSnJL2pJy8nJ0KJrYYUNJA5hd14LvVnu+Ati9MlizJ20+t6asQ1u/tIqgeGcuqquVd+8iu7SqVi33G3Fs3FfxsyeKlcVo7OoQ9vkj7CXRB82k72rfdpxrDB4HcRD0uNNf3OCHOyvfpVwHeXONnm+av2nnwnFp9nEByLjBiJePMPZoMpEI8wnuAEWE3MRo/+b3UnBbLTSCXF0ZAc1e9k3nebbcMAT/NZ8yE6AYIClhPp8Epsxz1fbJKz63S5Tv3DvatdtCUd4uFa+H7r89+ea3egU15sz8bBR/ExUxdmn1q1L+e6DpTrQXkzZFAg20Ji2MP191Fxy5v6NZdS9E3uuxDyIpiEQT0oz1c0M3/p6X9Rl7da/mMBBzRavxK4h4ae4Radwigtf80vjxkDKO2hPbddARJvH12napo+n01W99BZ2aY0quyB2+77ZvE5NnW/EOm9pvwezIE4uceM1No9Jv3uJV8lv+cmCp36oTciP2wTqPRXmjYdSC8Hn9uhwqMhuVY35o55t/51Y9+Qf/F1fs3gRjeAAA")))
open("/content/montage/logo_locapay.png", "wb").write(base64.b64decode("..."))
sys.path.insert(0, "/content/montage")
import importlib, montage_auto as M; importlib.reload(M)
import torch
print("✅ Installation terminée —", "GPU actif 🚀" if torch.cuda.is_available() else "⚠️ pas de GPU (ça marchera, mais plus lentement)")

In [ ]:
#@title 2️⃣ Réglages
#@markdown **Couleurs des écrans de couleur et de l'écran de fin**
couleur_1 = "#2E2C51" #@param {type:"string"}
couleur_2 = "#0AC2A4" #@param {type:"string"}
ecrans_de_couleur = 2 #@param {type:"slider", min:0, max:4, step:1}
#@markdown **Sous-titres et découpe**
position_sous_titres = 0.66 #@param {type:"slider", min:0.40, max:0.85, step:0.01}
couper_les_hesitations = True #@param {type:"boolean"}
pause_max_conservee = 0.45 #@param {type:"slider", min:0.20, max:1.00, step:0.05}
#@markdown **Son**
effets_sonores = "léger" #@param ["léger", "aucun"]
ajouter_une_musique = False #@param {type:"boolean"}
volume_musique = 0.18 #@param {type:"slider", min:0.05, max:0.50, step:0.01}
#@markdown **Logo** (cochez « autre_logo » pour charger un PNG à la place du logo Locapay)
logo = True #@param {type:"boolean"}
autre_logo = False #@param {type:"boolean"}
#@markdown **Transcription** — mots rares à bien reconnaître, séparés par des virgules
vocabulaire = "Locapay, démarcheurs, loyer, propriétaire, locataire, Cotonou, Bénin" #@param {type:"string"}
modele = "turbo" #@param ["turbo", "large-v3", "medium", "small"]
#@markdown **Vidéo déjà sur Google Drive ?** Indiquez son chemin (ex. `MyDrive/Videos/rush.mp4`), sinon laissez vide.
video_sur_drive = "" #@param {type:"string"}

R = M.Reglages(couleur_1=couleur_1, couleur_2=couleur_2, nb_ecrans_couleur=ecrans_de_couleur,
               position_sous_titres=position_sous_titres, supprimer_hesitations=couper_les_hesitations,
               pause_max=pause_max_conservee, effets_sonores=effets_sonores, volume_musique=volume_musique,
               logo_en_coin=logo, fichier_logo="/content/montage/logo_locapay.png" if logo else "",
               police_serif="/content/montage/polices/serif.ttf", police_grasse="/content/montage/polices/gras.ttf")
print("✅ Réglages enregistrés")

In [ ]:
#@title 3️⃣ Charger la vidéo (et la musique)
from google.colab import files
import shutil
os.makedirs("/content/montage/entree", exist_ok=True)
EXT_V = (".mp4", ".mov", ".m4v", ".avi", ".mkv", ".webm", ".3gp")
if video_sur_drive.strip():
    from google.colab import drive
    drive.mount("/content/drive")
    VIDEO = os.path.join("/content/drive", video_sur_drive.strip().lstrip("/"))
    assert os.path.exists(VIDEO), f"Fichier introuvable : {VIDEO}"
else:
    print("👉 Sélectionnez la vidéo brute :")
    up = files.upload()
    vids = [n for n in up if n.lower().endswith(EXT_V)]
    assert vids, "Aucune vidéo reconnue (formats : mp4, mov, m4v, avi, mkv, webm)."
    VIDEO = f"/content/montage/entree/{vids[0]}"
    shutil.move(vids[0], VIDEO)
MUSIQUE = None
if ajouter_une_musique:
    print("👉 Sélectionnez la musique (mp3, wav, m4a…) :")
    up = files.upload()
    if up:
        n = list(up)[0]; MUSIQUE = f"/content/montage/entree/{n}"; shutil.move(n, MUSIQUE)
if logo and autre_logo:
    print("👉 Sélectionnez votre logo (PNG, fond transparent de préférence) :")
    up = files.upload()
    if up:
        n = list(up)[0]; R.fichier_logo = f"/content/montage/entree/{n}"; shutil.move(n, R.fichier_logo)
DUREE = M.duree_video(VIDEO)
print(f"✅ Vidéo : {os.path.basename(VIDEO)} ({DUREE:.1f} s)" + (f" — musique : {os.path.basename(MUSIQUE)}" if MUSIQUE else ""))

In [ ]:
#@title 4️⃣ Transcription automatique
mots_bruts = M.transcrire(VIDEO, modele=modele, langue="fr", vocabulaire=vocabulaire)
print("\n📝 Transcription :\n")
print(M.texte_transcription(mots_bruts))

In [ ]:
#@title 5️⃣ Corrections (facultatif) et plan de montage
#@markdown Format : `mot_mal_reconnu=bon_mot; autre=correction` — ex. `Locapé=Locapay; démarcheur=démarcheurs`
corrections = "Locapé=Locapay; Lokapay=Locapay; Locapaï=Locapay" #@param {type:"string"}
mots = M.appliquer_corrections(mots_bruts, corrections)
PLAN = M.construire_plan(mots, DUREE, R)
print(M.resume_plan(PLAN))
print("\nLes mots en MAJUSCULES seront en gras. Relancez cette étape après une correction.")

In [ ]:
#@title 6️⃣ Montage et téléchargement
from IPython.display import Video, display
os.makedirs("/content/montage/sortie", exist_ok=True)
base = os.path.splitext(os.path.basename(VIDEO))[0]
SORTIE = f"/content/montage/sortie/{base}_monte.mp4"
print("🎬 Montage en cours…")
M.monter(VIDEO, PLAN, R, SORTIE, musique=MUSIQUE)
M.ecrire_srt(PLAN, SORTIE.replace(".mp4", ".srt"))
open(SORTIE.replace(".mp4", "_transcription.txt"), "w", encoding="utf-8").write(M.texte_transcription(mots))
print("✅ Terminé !")
display(Video(SORTIE, embed=True, width=320))
files.download(SORTIE)

---
**Fichiers produits** (dossier `/content/montage/sortie`, panneau 📁 à gauche) : la vidéo montée, les sous-titres `.srt` et la transcription `.txt`.

**Astuces**
- Filmez en vertical, la personne au centre, avec un son propre : la transcription n'en sera que meilleure.
- Trop de coupes ? Montez « pause_max_conservee » (étape 2). Pas assez ? Baissez-la.
- Sous-titres qui cachent les mains ou le visage ? Ajustez « position_sous_titres ».